In [ ]:
# 设置环境变量
%env HF_ENDPOINT=https://hf-mirror.com
%env HF_HOME=/root/autodl-tmp/hf

In [ ]:
# 加载model和tkennizer
from transformers import AutoModelForCausalLM,AutoTokenizer
import torch
from peft import PeftModel

# 模型名称
model_name = "Qwen/Qwen3-8B"
# LoRA微调之后的模型Adapter
lora_adpter = "/root/autodl-tmp/sft/Qwen3-8B/LoRA/best"

# 基础模型
base_model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16)
# 增加LoRA微调adpater
peft_model = PeftModel.from_pretrained(base_model, lora_adpter, torch_dtype=torch.float16)
# 模型合并
mergered_model = peft_model.merge_and_unload()

tokenizer = AutoTokenizer.from_pretrained(model_name)


In [ ]:
# prepare the model input
prompt = "抽取出文本中的关键词：\n标题：人工神经网络在猕猴桃种类识别上的应用\n文本：在猕猴桃介电特性研究的基础上,将人工神经网络技术应用于猕猴桃的种类识别.该种类识别属于模式识别,其关键在于提取样品的特征参数,在获得特征参数的基础上,选取合适的网络通过训练来进行识别.猕猴桃种类识别的研究为自动化识别果品的种类、品种和新鲜等级等提供了一种新方法,为进一步研究果品介电特性与其内在品质的关系提供了一定的理论与实践基础."
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False # Switches between thinking and non-thinking modes. Default is True.
)
print(text)

In [ ]:
# 模型输入
model_inputs = tokenizer([text], return_tensors="pt").to(mergered_model.device)

# conduct text completion
generated_ids = mergered_model.generate(
    **model_inputs,
    max_new_tokens=32768
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

# parsing thinking content
try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("thinking content:", thinking_content)
print("content:", content)

# 微调前的输出
# content: 关键词：人工神经网络、猕猴桃、种类识别、模式识别、特征参数、果品、介电特性、品质、研究方法。

# 微调后的输出
# content: 猕猴桃;人工神经网络;种类识别;特征参数;介电特性